&nbsp;

&nbsp;

&nbsp;

&nbsp;

# GENCODE Release Compare (release 46, 49)

- どの lncRNA が新しく release 47 から追加されたのかを調査しておく
- CLS project というタグだけでは必ずしも判別できない

**Workflow**

1. GENCODE v46 / v49 の GTF を読み込む
2. ID ベースで v49 の transcript を分類する (`novelty_level`)
3. `gffcompare` の結果 (構造ベースの `class_code`) を対応させる
4. 06_6 で使うテーブルを保存する

In [1]:
import polars as pl
import datetime
import pathlib

from IPython.display import display

pl.Config.set_fmt_str_lengths(100)

polars.config.Config

In [2]:
VERSION = datetime.datetime.now().strftime("%y%m")
Execute_date = datetime.datetime.now().strftime("%y%m%d")
print(f"Execute Date: {Execute_date}")

Execute Date: 261110


&nbsp;

&nbsp;

&nbsp;

## 1. Load GTF files

&nbsp;

### 1-1. GENCODE release 46 (GTF file)

- Download date: 2026-11-10

In [3]:
# For more information, see https://www.gencodegenes.org/pages/biotypes.html
gencode_v46 = pl.read_csv(
    "../../Data/Database/GENCODE_v46/gencode.v46.chr_patch_hapl_scaff.annotation.gtf",
    separator='\t',
    has_header=False,
    comment_prefix='#',
    new_columns=['seqname', 'source', 'feature', 'start', 'end', 'score', 'strand', 'frame', 'attributes'],
    n_threads=16
).filter(
    pl.col('feature') == 'transcript'
).with_columns(
    # Unique keys: extract the first (and only) quoted value
    pl.col("attributes").str.extract(r'gene_id "([^"]+)"', 1).alias("gene_id"),
    pl.col("attributes").str.extract(r'transcript_id "([^"]+)"', 1).alias("transcript_id"),
    pl.col("attributes").str.extract(r'transcript_type "([^"]+)"', 1).alias("transcript_type"),
    # tag can appear multiple times; extract_all returns full matches like tag "basic"
    pl.col("attributes")
        .str.extract_all(r'tag "[^"]+"')
        .list.eval(pl.element().str.extract(r'"([^"]+)"', 1))
        .alias("tags")
)

pl.Config.set_fmt_str_lengths(500)
pl.Config.set_tbl_rows(2)
display(gencode_v46)

seqname,source,feature,start,end,score,strand,frame,attributes,gene_id,transcript_id,transcript_type,tags
str,str,str,i64,i64,str,str,str,str,str,str,str,list[str]
"""chr1""","""HAVANA""","""transcript""",11869,14409,""".""","""+""",""".""","""gene_id ""ENSG00000290825.1""; transcript_id ""ENST00000456328.2""; gene_type ""lncRNA""; gene_name ""DDX11L2""; transcript_type ""lncRNA""; transcript_name ""DDX11L2-202""; level 2; transcript_support_level ""1""; tag ""basic""; tag ""Ensembl_canonical""; havana_transcript ""OTTHUMT00000362751.1"";""","""ENSG00000290825.1""","""ENST00000456328.2""","""lncRNA""","[""basic"", ""Ensembl_canonical""]"
…,…,…,…,…,…,…,…,…,…,…,…,…
"""KI270750.1""","""ENSEMBL""","""transcript""",148668,148843,""".""","""+""",""".""","""gene_id ""ENSG00000277374.1""; transcript_id ""ENST00000612925.1""; gene_type ""snRNA""; gene_name ""U1""; transcript_type ""snRNA""; transcript_name ""U1.29-201""; level 3; transcript_support_level ""NA""; tag ""basic""; tag ""Ensembl_canonical"";""","""ENSG00000277374.1""","""ENST00000612925.1""","""snRNA""","[""basic"", ""Ensembl_canonical""]"


&nbsp;

### 1-2. GENCODE release 49 (GTF file)

In [4]:
# For more information, see https://www.gencodegenes.org/pages/biotypes.html
gencode_v49 = pl.read_csv(
    "../../Data/Database/GENCODE_v49/gencode.v49.chr_patch_hapl_scaff.annotation.gtf",
    separator='\t',
    has_header=False,
    comment_prefix='#',
    new_columns=['seqname', 'source', 'feature', 'start', 'end', 'score', 'strand', 'frame', 'attributes'],
    n_threads=16
).filter(
    pl.col('feature') == 'transcript'
).with_columns(
    # Unique keys: extract the first (and only) quoted value
    pl.col("attributes").str.extract(r'gene_id "([^"]+)"', 1).alias("gene_id"),
    pl.col("attributes").str.extract(r'transcript_id "([^"]+)"', 1).alias("transcript_id"),
    pl.col("attributes").str.extract(r'transcript_type "([^"]+)"', 1).alias("transcript_type"),
    # tag can appear multiple times; extract_all returns full matches like tag "basic"
    pl.col("attributes")
        .str.extract_all(r'tag "[^"]+"')
        .list.eval(pl.element().str.extract(r'"([^"]+)"', 1))
        .alias("tags")
)

pl.Config.set_fmt_str_lengths(500)
pl.Config.set_tbl_rows(2)
display(gencode_v49)

seqname,source,feature,start,end,score,strand,frame,attributes,gene_id,transcript_id,transcript_type,tags
str,str,str,i64,i64,str,str,str,str,str,str,str,list[str]
"""chr1""","""HAVANA""","""transcript""",11121,14413,""".""","""+""",""".""","""gene_id ""ENSG00000290825.2""; transcript_id ""ENST00000832824.1""; gene_type ""lncRNA""; gene_name ""DDX11L16""; transcript_type ""lncRNA""; transcript_name ""DDX11L16-260""; level 2; tag ""TAGENE"";""","""ENSG00000290825.2""","""ENST00000832824.1""","""lncRNA""","[""TAGENE""]"
…,…,…,…,…,…,…,…,…,…,…,…,…
"""KI270755.1""","""HAVANA""","""transcript""",26350,27723,""".""","""+""",""".""","""gene_id ""ENSG00000309258.1""; transcript_id ""ENST00000839853.1""; gene_type ""lncRNA""; gene_name ""ENSG00000309258""; transcript_type ""lncRNA""; transcript_name ""ENST00000839853""; level 2; tag ""basic""; tag ""Ensembl_canonical""; tag ""GENCODE_Primary""; tag ""TAGENE"";""","""ENSG00000309258.1""","""ENST00000839853.1""","""lncRNA""","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]"


&nbsp;

&nbsp;

&nbsp;

## 2. ID-based classification

&nbsp;

### 2-1. Strip version suffix

- `ENST00000456328.2` → `ENST00000456328` とし、バージョン番号の更新だけで「新規」と数えないようにする

In [5]:
# Strip the version suffix (e.g. ENST00000456328.2 -> ENST00000456328)
# so that version bumps of existing IDs are not counted as new
def add_base_ids(df: pl.DataFrame) -> pl.DataFrame:
    return df.with_columns(
        pl.col("gene_id").str.replace(r"\.\d+", "").alias("gene_id_base"),
        pl.col("transcript_id").str.replace(r"\.\d+", "").alias("transcript_id_base"),
    )

gencode_v46_base = add_base_ids(gencode_v46)
gencode_v49_base = add_base_ids(gencode_v49)

&nbsp;

### 2-2. (Simple version) Transcripts in v49 but not in v46

- 遺伝子 ID が v46 にあるかどうかだけで `new_gene` / `new_transcript_in_existing_gene` を判定した簡易版
- 古い transcript が新しい遺伝子 ID に付け替えられたケースも `new_gene` に入ってしまう → 2-3 で厳密版を作る

In [6]:
# Transcripts in v49 but not in v46, labelled by whether the gene already existed in v46
gencode_v49_new_transcript = (
    gencode_v49_base
    .join(gencode_v46_base.select("transcript_id_base"), on="transcript_id_base", how="anti")
    .with_columns(
        pl.col("gene_id_base").is_in(gencode_v46_base["gene_id_base"].implode()).alias("is_gene_in_v46")
    )
    .with_columns(
        pl.when(pl.col("is_gene_in_v46"))
        .then(pl.lit("new_transcript_in_existing_gene"))
        .otherwise(pl.lit("new_gene"))
        .alias("novelty_level")
    )
)

display(gencode_v49_new_transcript)

seqname,source,feature,start,end,score,strand,frame,attributes,gene_id,transcript_id,transcript_type,tags,gene_id_base,transcript_id_base,is_gene_in_v46,novelty_level
str,str,str,i64,i64,str,str,str,str,str,str,str,list[str],str,str,bool,str
"""chr1""","""HAVANA""","""transcript""",11121,14413,""".""","""+""",""".""","""gene_id ""ENSG00000290825.2""; transcript_id ""ENST00000832824.1""; gene_type ""lncRNA""; gene_name ""DDX11L16""; transcript_type ""lncRNA""; transcript_name ""DDX11L16-260""; level 2; tag ""TAGENE"";""","""ENSG00000290825.2""","""ENST00000832824.1""","""lncRNA""","[""TAGENE""]","""ENSG00000290825""","""ENST00000832824""",true,"""new_transcript_in_existing_gene"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""KI270755.1""","""HAVANA""","""transcript""",26350,27723,""".""","""+""",""".""","""gene_id ""ENSG00000309258.1""; transcript_id ""ENST00000839853.1""; gene_type ""lncRNA""; gene_name ""ENSG00000309258""; transcript_type ""lncRNA""; transcript_name ""ENST00000839853""; level 2; tag ""basic""; tag ""Ensembl_canonical""; tag ""GENCODE_Primary""; tag ""TAGENE"";""","""ENSG00000309258.1""","""ENST00000839853.1""","""lncRNA""","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]","""ENSG00000309258""","""ENST00000839853""",false,"""new_gene"""


In [7]:
print(f"New transcripts in v49: {gencode_v49_new_transcript.height}")

pl.Config.set_tbl_rows(10)
display(
    gencode_v49_new_transcript
    .group_by("novelty_level", "transcript_type")
    .agg(
        pl.len().alias("n_transcript"),
        pl.col("gene_id_base").n_unique().alias("n_gene"),
    )
    .sort("n_transcript", descending=True)
)

New transcripts in v49: 256028


novelty_level,transcript_type,n_transcript,n_gene
str,str,u32,u32
"""new_transcript_in_existing_gene""","""protein_coding""",122087,13104
"""new_transcript_in_existing_gene""","""lncRNA""",87509,8671
"""new_gene""","""lncRNA""",46128,16833
"""new_transcript_in_existing_gene""","""nonsense_mediated_decay""",117,79
"""new_transcript_in_existing_gene""","""protein_coding_CDS_not_defined""",80,47
…,…,…,…
"""new_transcript_in_existing_gene""","""transcribed_unitary_pseudogene""",1,1
"""new_gene""","""unitary_pseudogene""",1,1
"""new_transcript_in_existing_gene""","""non_stop_decay""",1,1


&nbsp;

#### Check: example transcript

In [8]:
ENST00000778743 = gencode_v49_new_transcript.filter(
    pl.col("transcript_id") == "ENST00000778743.1"
)

display(ENST00000778743)

seqname,source,feature,start,end,score,strand,frame,attributes,gene_id,transcript_id,transcript_type,tags,gene_id_base,transcript_id_base,is_gene_in_v46,novelty_level
str,str,str,i64,i64,str,str,str,str,str,str,str,list[str],str,str,bool,str
"""chr17""","""HAVANA""","""transcript""",68052662,68053305,""".""","""+""",""".""","""gene_id ""ENSG00000301409.1""; transcript_id ""ENST00000778743.1""; gene_type ""lncRNA""; gene_name ""ENSG00000301409""; transcript_type ""lncRNA""; transcript_name ""ENST00000778743""; level 2; tag ""basic""; tag ""Ensembl_canonical""; tag ""GENCODE_Primary""; tag ""TAGENE"";""","""ENSG00000301409.1""","""ENST00000778743.1""","""lncRNA""","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]","""ENSG00000301409""","""ENST00000778743""",false,"""new_gene"""


&nbsp;

### 2-3. (Strict version) Novelty level of all v49 transcripts

- `existing_in_v46`: transcript already present in v46
- `new_transcript_in_existing_gene`: new transcript added to a gene present in v46
- `new_gene`: gene ID absent in v46 **and** none of its transcripts existed in v46 (excludes re-assigned old transcripts)

&nbsp;

#### Step 1: v46 に transcript ID / gene ID があるかを判定

In [9]:
# Flag whether each v49 transcript ID and gene ID already existed in v46
gencode_v49_in_v46_flag = gencode_v49_base.with_columns(
    pl.col("transcript_id_base").is_in(gencode_v46_base["transcript_id_base"].implode()).alias("is_transcript_in_v46"),
    pl.col("gene_id_base").is_in(gencode_v46_base["gene_id_base"].implode()).alias("is_gene_in_v46"),
)

display(gencode_v49_in_v46_flag.select("transcript_id", "gene_id", "is_transcript_in_v46", "is_gene_in_v46"))

transcript_id,gene_id,is_transcript_in_v46,is_gene_in_v46
str,str,bool,bool
"""ENST00000832824.1""","""ENSG00000290825.2""",false,true
"""ENST00000832825.1""","""ENSG00000290825.2""",false,true
"""ENST00000832826.1""","""ENSG00000290825.2""",false,true
"""ENST00000832827.1""","""ENSG00000290825.2""",false,true
"""ENST00000832828.1""","""ENSG00000290825.2""",false,true
…,…,…,…
"""ENST00000797959.1""","""ENSG00000303902.1""",false,false
"""ENST00000819228.1""","""ENSG00000306528.1""",false,false
"""ENST00000819229.1""","""ENSG00000306528.1""",false,false


&nbsp;

#### Step 2: 遺伝子ごと新規かどうかを判定

In [10]:
# A gene is new only if its ID is absent in v46 and none of its transcripts existed in v46
gencode_v49_new_gene_flag = gencode_v49_in_v46_flag.with_columns(
    (
        ~pl.col("is_gene_in_v46") 
        & ~pl.col("is_transcript_in_v46").any().over("gene_id_base")
    ).alias("is_new_gene")
)

display(gencode_v49_new_gene_flag.select("transcript_id", "gene_id", "is_transcript_in_v46", "is_gene_in_v46", "is_new_gene"))

transcript_id,gene_id,is_transcript_in_v46,is_gene_in_v46,is_new_gene
str,str,bool,bool,bool
"""ENST00000832824.1""","""ENSG00000290825.2""",false,true,false
"""ENST00000832825.1""","""ENSG00000290825.2""",false,true,false
"""ENST00000832826.1""","""ENSG00000290825.2""",false,true,false
"""ENST00000832827.1""","""ENSG00000290825.2""",false,true,false
"""ENST00000832828.1""","""ENSG00000290825.2""",false,true,false
…,…,…,…,…
"""ENST00000797959.1""","""ENSG00000303902.1""",false,false,true
"""ENST00000819228.1""","""ENSG00000306528.1""",false,false,true
"""ENST00000819229.1""","""ENSG00000306528.1""",false,false,true


&nbsp;

#### Step 3: `novelty_level` を付ける

In [11]:
# Per-transcript novelty of all GENCODE v49 transcripts relative to v46 (used in 06_6)
gencode_v49_novelty = (
    gencode_v49_new_gene_flag
    .with_columns(
        pl.when(pl.col("is_transcript_in_v46")).then(pl.lit("existing_in_v46"))
        .when(pl.col("is_new_gene")).then(pl.lit("new_gene"))
        .otherwise(pl.lit("new_transcript_in_existing_gene"))
        .alias("novelty_level")
    )
    .select("transcript_id", "gene_id", "transcript_type", "novelty_level")
)

display(gencode_v49_novelty)

transcript_id,gene_id,transcript_type,novelty_level
str,str,str,str
"""ENST00000832824.1""","""ENSG00000290825.2""","""lncRNA""","""new_transcript_in_existing_gene"""
"""ENST00000832825.1""","""ENSG00000290825.2""","""lncRNA""","""new_transcript_in_existing_gene"""
"""ENST00000832826.1""","""ENSG00000290825.2""","""lncRNA""","""new_transcript_in_existing_gene"""
"""ENST00000832827.1""","""ENSG00000290825.2""","""lncRNA""","""new_transcript_in_existing_gene"""
"""ENST00000832828.1""","""ENSG00000290825.2""","""lncRNA""","""new_transcript_in_existing_gene"""
…,…,…,…
"""ENST00000797959.1""","""ENSG00000303902.1""","""lncRNA""","""new_gene"""
"""ENST00000819228.1""","""ENSG00000306528.1""","""lncRNA""","""new_gene"""
"""ENST00000819229.1""","""ENSG00000306528.1""","""lncRNA""","""new_gene"""


&nbsp;

#### Step 4: `novelty_level` ごとの件数

In [12]:
print(f"GENCODE v49 transcripts: {gencode_v49_novelty.height}")

pl.Config.set_tbl_rows(10)
display(
    gencode_v49_novelty
    .group_by("novelty_level")
    .agg(
        pl.len().alias("n_transcript"),
        pl.col("gene_id").n_unique().alias("n_gene"),
    )
    .sort("novelty_level")
)

GENCODE v49 transcripts: 533740


novelty_level,n_transcript,n_gene
str,u32,u32
"""existing_in_v46""",277712,69508
"""new_gene""",45690,16836
"""new_transcript_in_existing_gene""",210338,21821


&nbsp;

&nbsp;

&nbsp;

## 3. Structure-based comparison (`gffcompare`)

&nbsp;

### 3-1. Load `gffcompare` result (`.tracking`)

- `cwltool --outdir ./out/gencode_release_compare/ ./Tools/fantomcat/gffcompare.cwl ./config/gffcompare/gencode_compare_v46_v49.yml`
- reference: v46, query: v49
- intron chain が他の v49 transcript と同じ冗長な transcript (3,254 本) は gffcompare が 1 本にまとめるため、`.tracking` には載らない

In [13]:
gffcompare_result_v46_49 = pl.read_csv(
    "../../out/gencode_release_compare/gencode_compare_v46_v49.tracking",
    separator="\t",
    has_header=False,
    new_columns=[
        "Entry",
        "Super_locus",
        "gencode_v46",
        "class_code",
        "gencode_v49"
    ]
)

display(gffcompare_result_v46_49)

Entry,Super_locus,gencode_v46,class_code,gencode_v49
str,str,str,str,str
"""gencode_compare_v46_v49_00000001|3|1149""","""XLOC_000001""","""-""","""u""","""q1:ENSG00000296732.1|ENST00000741402.1|3|0.000000|0.000000|0.000000|1149"""
"""gencode_compare_v46_v49_00000002|3|914""","""XLOC_000001""","""-""","""u""","""q1:ENSG00000296732.1|ENST00000741435.1|3|0.000000|0.000000|0.000000|914"""
"""gencode_compare_v46_v49_00000003|4|903""","""XLOC_000001""","""-""","""u""","""q1:ENSG00000296732.1|ENST00000741407.1|4|0.000000|0.000000|0.000000|903"""
"""gencode_compare_v46_v49_00000004|3|1482""","""XLOC_000001""","""-""","""u""","""q1:ENSG00000296732.1|ENST00000741436.1|3|0.000000|0.000000|0.000000|1482"""
"""gencode_compare_v46_v49_00000005|3|955""","""XLOC_000001""","""-""","""u""","""q1:ENSG00000296732.1|ENST00000741404.1|3|0.000000|0.000000|0.000000|955"""
…,…,…,…,…
"""gencode_compare_v46_v49_00530482|4|1323""","""XLOC_078428""","""ENSG00000292371.1|ENST00000711270.1""","""j""","""q1:ENSG00000310542.1|ENST00000972821.1|4|0.000000|0.000000|0.000000|1323"""
"""gencode_compare_v46_v49_00530483|5|1306""","""XLOC_078428""","""ENSG00000292371.1|ENST00000711270.1""","""=""","""q1:ENSG00000292371.1|ENST00000711270.1|5|0.000000|0.000000|0.000000|1306"""
"""gencode_compare_v46_v49_00530484|4|1241""","""XLOC_078428""","""ENSG00000292371.1|ENST00000711270.1""","""j""","""q1:ENSG00000310542.1|ENST00000972824.1|4|0.000000|0.000000|0.000000|1241"""


&nbsp;

### 3-2. Extract transcript IDs from the `.tracking` columns

- `gencode_v49`: `q1:<gene_id>|<transcript_id>|<num_exons>|...`
- `gencode_v46`: `<gene_id>|<transcript_id>` (一致する reference がなければ `-`)

In [14]:
# Extract v49 transcript ID and matched v46 reference IDs from the tracking file
gffcompare_class_v49 = (
    gffcompare_result_v46_49
    .with_columns(
        pl.col("gencode_v49").str.split("|").list.get(1).alias("transcript_id"),
        pl.col("gencode_v46").replace("-", None).str.split("|").list.get(0).alias("ref_gene_id_v46"),
        pl.col("gencode_v46").replace("-", None).str.split("|").list.get(1).alias("ref_transcript_id_v46"),
    )
    .select("transcript_id", "class_code", "ref_gene_id_v46", "ref_transcript_id_v46")
)

display(gffcompare_class_v49)

transcript_id,class_code,ref_gene_id_v46,ref_transcript_id_v46
str,str,str,str
"""ENST00000741402.1""","""u""",null,null
"""ENST00000741435.1""","""u""",null,null
"""ENST00000741407.1""","""u""",null,null
"""ENST00000741436.1""","""u""",null,null
"""ENST00000741404.1""","""u""",null,null
…,…,…,…
"""ENST00000972821.1""","""j""","""ENSG00000292371.1""","""ENST00000711270.1"""
"""ENST00000711270.1""","""=""","""ENSG00000292371.1""","""ENST00000711270.1"""
"""ENST00000972824.1""","""j""","""ENSG00000292371.1""","""ENST00000711270.1"""


&nbsp;

### 3-3. Join `class_code` to the novelty table (2-3)

- `.tracking` に載らない transcript は `class_code` = `not_in_tracking` とする

In [15]:
# Add gffcompare class codes to the ID-based novelty table
gencode_v49_novelty_gffcompare = (
    gencode_v49_novelty
    .join(gffcompare_class_v49, on="transcript_id", how="left")
    .with_columns(pl.col("class_code").fill_null("not_in_tracking"))
)

display(gencode_v49_novelty_gffcompare)

transcript_id,gene_id,transcript_type,novelty_level,class_code,ref_gene_id_v46,ref_transcript_id_v46
str,str,str,str,str,str,str
"""ENST00000832824.1""","""ENSG00000290825.2""","""lncRNA""","""new_transcript_in_existing_gene""","""j""","""ENSG00000290825.1""","""ENST00000456328.2"""
"""ENST00000832825.1""","""ENSG00000290825.2""","""lncRNA""","""new_transcript_in_existing_gene""","""j""","""ENSG00000290825.1""","""ENST00000456328.2"""
"""ENST00000832826.1""","""ENSG00000290825.2""","""lncRNA""","""new_transcript_in_existing_gene""","""j""","""ENSG00000290825.1""","""ENST00000456328.2"""
"""ENST00000832827.1""","""ENSG00000290825.2""","""lncRNA""","""new_transcript_in_existing_gene""","""j""","""ENSG00000290825.1""","""ENST00000456328.2"""
"""ENST00000832828.1""","""ENSG00000290825.2""","""lncRNA""","""new_transcript_in_existing_gene""","""j""","""ENSG00000290825.1""","""ENST00000456328.2"""
…,…,…,…,…,…,…
"""ENST00000797959.1""","""ENSG00000303902.1""","""lncRNA""","""new_gene""","""u""",null,null
"""ENST00000819228.1""","""ENSG00000306528.1""","""lncRNA""","""new_gene""","""u""",null,null
"""ENST00000819229.1""","""ENSG00000306528.1""","""lncRNA""","""new_gene""","""u""",null,null


&nbsp;

### 3-4. Cross-tabulation: `class_code` × `novelty_level`

In [16]:
# Count transcripts for each combination of structural class code and ID-based novelty
pl.Config.set_tbl_rows(20)
display(
    gencode_v49_novelty_gffcompare
    .pivot(on="novelty_level", index="class_code", values="transcript_id", aggregate_function="len")
    .fill_null(0)
    .sort("class_code")
)

class_code,new_transcript_in_existing_gene,existing_in_v46,new_gene
str,u32,u32,u32
"""=""",551,272253,44
"""c""",66,10,56
"""e""",545,1,6
"""i""",974,0,6214
"""j""",188104,67,4407
"""k""",7770,2166,925
"""m""",2028,3,52
"""n""",1837,1,354
"""not_in_tracking""",40,3204,10


&nbsp;

&nbsp;

&nbsp;

## 4. Export (used in 06_6)

In [17]:
# Output directory
out_dir = pathlib.Path("../../out/gencode_release_compare")
out_dir.mkdir(parents=True, exist_ok=True)

&nbsp;

### 4-1. ID-based novelty table

In [18]:
gencode_v49_novelty.write_parquet(
    out_dir / f"gencode_v49_transcript_novelty_vs_v46_v{Execute_date}.parquet"
)

&nbsp;

### 4-2. ID-based novelty + `gffcompare` class code table

In [19]:
gencode_v49_novelty_gffcompare.write_parquet(
    out_dir / f"gencode_v49_transcript_novelty_gffcompare_vs_v46_v{Execute_date}.parquet"
)